# Kaggle 2 NLP


In [ ]:
import os
import sys
import re
import gc
import ast
import html
import json
import math
import time
import shutil
import random
import warnings
import tempfile
from pathlib import Path
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")

STUDENT_ID = 322

DATA_DIR = Path(r"D:\ДЛ РАНХ\кагл 2")

ROOT_DIR = Path(r"D:\dl_kaggle_322_v6")
WORK_DIR = ROOT_DIR / "work"
CACHE_DIR = ROOT_DIR / "cache"
TMP_DIR = ROOT_DIR / "tmp"
MODEL_DIR = ROOT_DIR / "models"

HF_HOME = CACHE_DIR / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
HF_DATASETS_CACHE = HF_HOME / "datasets"
TORCH_HOME = CACHE_DIR / "torch"
PIP_CACHE_DIR = CACHE_DIR / "pip"

for p in [
    DATA_DIR, ROOT_DIR, WORK_DIR, CACHE_DIR, TMP_DIR, MODEL_DIR,
    HF_HOME, HF_HUB_CACHE, TRANSFORMERS_CACHE, HF_DATASETS_CACHE,
    TORCH_HOME, PIP_CACHE_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)
os.environ["TORCH_HOME"] = str(TORCH_HOME)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
os.environ["TMP"] = str(TMP_DIR)
os.environ["TEMP"] = str(TMP_DIR)
os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
tempfile.tempdir = str(TMP_DIR)

def seed_everything(seed=322):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except Exception:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

seed_everything(STUDENT_ID)

def gb(x):
    return x / 1024**3

total, used, free = shutil.disk_usage(ROOT_DIR)
print("DATA_DIR:", DATA_DIR)
print("ROOT_DIR:", ROOT_DIR)
print("HF_HUB_CACHE:", HF_HUB_CACHE)
print(f"D: total={gb(total):.2f} GB | used={gb(used):.2f} GB | free={gb(free):.2f} GB")

DATA_DIR: D:\ДЛ РАНХ\кагл 2
ROOT_DIR: D:\dl_kaggle_322_v6
HF_HUB_CACHE: D:\dl_kaggle_322_v6\cache\huggingface\hub
D: total=930.91 GB | used=676.72 GB | free=254.19 GB


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.naive_bayes import ComplementNB
from scipy import sparse

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch недоступен:", repr(e))

try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False
    print("Transformers недоступен, HF-модели будут пропущены:", repr(e))

CFG = {
    "seed": 322,
    "n_labels": 5,

    "use_hf": True,
    "use_textcnn": True,
    "use_sparse": True,

    "hf_models": [
        {
            "name": "cointegrated/rubert-tiny2",
            "tag": "rubert_tiny2",
            "epochs": 4,
            "max_length": 384,
            "batch_size": 16,
            "valid_batch_size": 32,
            "lr": 2e-5,
            "weight_decay": 0.01,
            "use_safetensors": True,
            "revision": None,
        },
        {
            "name": "FacebookAI/xlm-roberta-base",
            "tag": "xlm_roberta_base",
            "epochs": 3,
            "max_length": 384,
            "batch_size": 8,
            "valid_batch_size": 16,
            "lr": 1.5e-5,
            "weight_decay": 0.01,
            "use_safetensors": True,
            "revision": None,
        },
    ],


    "textcnn_epochs": 6,
    "textcnn_batch_size": 32,
    "textcnn_max_len": 700,
    "textcnn_vocab_size": 70000,
    "textcnn_emb_dim": 256,
    "textcnn_lr": 1e-3,

    "out_csv": str(DATA_DIR / "sample_submission.csv"),
    "extra_out_csv": str(DATA_DIR / "sample_submission_322_v6.csv"),
}

print(json.dumps({k:v for k,v in CFG.items() if k!="hf_models"}, ensure_ascii=False, indent=2))
print("TORCH_OK:", TORCH_OK, "TRANSFORMERS_OK:", TRANSFORMERS_OK)
if TORCH_OK:
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available())
if TRANSFORMERS_OK:
    print("transformers:", transformers.__version__)

{
  "seed": 322,
  "n_labels": 5,
  "use_hf": true,
  "use_textcnn": true,
  "use_sparse": true,
  "textcnn_epochs": 6,
  "textcnn_batch_size": 32,
  "textcnn_max_len": 700,
  "textcnn_vocab_size": 70000,
  "textcnn_emb_dim": 256,
  "textcnn_lr": 0.001,
  "out_csv": "D:\\ДЛ РАНХ\\кагл 2\\sample_submission.csv",
  "extra_out_csv": "D:\\ДЛ РАНХ\\кагл 2\\sample_submission_322_v6.csv"
}
TORCH_OK: True TRANSFORMERS_OK: True
torch: 2.5.1+cu121
cuda: True
transformers: 5.8.1


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

for path in [train_path, test_path, sample_path]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

train = pd.read_csv(train_path, sep="\t")
test = pd.read_csv(test_path, sep="\t")
sample = pd.read_csv(sample_path)

print("train:", train.shape)
print("test:", test.shape)
print("sample:", sample.shape)
display(train.head(2))
display(test.head(2))
display(sample.head(2))

train: (16701, 6)
test: (4969, 5)
sample: (4969, 2)


,id,source,title,text,publication_date,target
0,0,Novosti,Рейтинг регионов по уровню закредитованности н...,Средний <content>уровень</content> <source>ria...,2019-12-23 00:00,"[0, 1, 0, 0, 0]"
1,1,Novosti,Названы самые закредитованные российские регионы,"МОСКВА, 23 дек — РИА Новости. Наиболее закре...",2019-12-23 00:21,"[0, 1, 0, 0, 0]"


,id,source,title,text,publication_date
0,16701,Spletnesti,Дым от австралийс...,...,2020-01-01 07:35
1,16702,Spletnesti,Во Владивостоке в...,...,2020-01-01 08:22


,id,target
0,16701,"[0,0,0,0,0]"
1,16702,"[0,0,0,0,0]"


In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
CDATA_RE = re.compile(r"<!\[CDATA\[|\]\]>")
XML_RE = re.compile(r"<\?xml[^>]*\?>", flags=re.I)
COMMENT_RE = re.compile(r"<!--.*?-->", flags=re.S)
URL_RE = re.compile(r"(https?://\S+|www\.\S+|<url>)", flags=re.I)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
BAD_TAG_WORDS_RE = re.compile(
    r"\b(content|source|section|article|blockquote|strong|span|class|quote|text|body|div|li|ul|hr|br|em|xml|encoding|utf|wp|paragraph|doc|news|item|time)\b",
    flags=re.I,
)

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

PUNCT_MAP = str.maketrans({
    "«": " ", "»": " ", "“": " ", "”": " ", "„": " ", "’": " ", "‘": " ",
    "—": " ", "–": " ", "−": " ", "…": " ", "\xa0": " ",
})

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = html.unescape(x)
    x = x.translate(PUNCT_MAP)
    x = XML_RE.sub(" ", x)
    x = COMMENT_RE.sub(" ", x)
    x = CDATA_RE.sub(" ", x)
    x = URL_RE.sub(" url ", x)
    x = EMAIL_RE.sub(" email ", x)
    x = HTML_RE.sub(" ", x)
    x = EMOJI_RE.sub(" ", x)
    x = x.replace("\\n", " ").replace("\\r", " ").replace("\\t", " ")
    x = re.sub(r"&[a-zA-Z]+;", " ", x)
    x = BAD_TAG_WORDS_RE.sub(" ", x)
    x = re.sub(r"[^0-9A-Za-zА-Яа-яЁё%.,:;!?()\-+/ ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

def get_year_month(s):
    try:
        dt = pd.to_datetime(s, errors="coerce")
        if pd.isna(dt):
            return "year_unk month_unk"
        return f"year_{dt.year} month_{dt.month:02d}"
    except Exception:
        return "year_unk month_unk"

def build_features(df):
    out = pd.DataFrame(index=df.index)
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)
    source = df["source"].fillna("unknown").astype(str).str.lower().str.replace(r"\W+", "_", regex=True)
    date_tokens = df["publication_date"].fillna("").map(get_year_month)

    out["clean_title"] = title
    out["clean_text"] = text
    out["source_token"] = " src_" + source
    out["date_token"] = " " + date_tokens

    out["full_text"] = (
        " title_title " + title + 
        " title_title " + title + 
        " body_body " + text +
        out["source_token"] +
        out["date_token"]
    ).str.strip()

    out["short_text"] = (
        " title_title " + title + 
        " body_start " + text.str.slice(0, 2500) +
        out["source_token"] +
        out["date_token"]
    ).str.strip()

    out["title_only"] = (" title_title " + title + out["source_token"]).str.strip()
    return out

train_feat = build_features(train)
test_feat = build_features(test)

def parse_target(x):
    if isinstance(x, list):
        return x
    return list(ast.literal_eval(str(x).replace(" ", "")))

Y = np.array(train["target"].map(parse_target).tolist(), dtype=np.int64)
label_key = np.array(["".join(map(str, row)) for row in Y])

print("Y:", Y.shape)
print("label distribution:")
print(pd.Series(label_key).value_counts().head(20))
print("clean example:")
print(train_feat["full_text"].iloc[0][:1000])

Y: (16701, 5)
label distribution:
10000    5766
00000    5643
00100    1455
01000    1169
11000     689
00010     516
00001     303
10010     301
01010     246
10100     171
10001      95
11010      90
00101      90
00110      60
01100      28
01001      23
11100      13
00011      11
11001       9
01110       5
Name: count, dtype: int64
clean example:
title_title рейтинг регионов по уровню закредитованности населения 2019 title_title рейтинг регионов по уровню закредитованности населения 2019 body_body средний уровень ria.ru закредитованности россиян вырос за 2019 год с 44,9 copy;до 47,1 %. u больше всего банкам url должны жители p калмыкии (86, 2 %), меньше всего ингушетии (9,9%). смотрите в инфографике ria.ru, какая долговая нагрузка у населения вашего региона src_novosti year_2019 month_12


In [ ]:
vc = pd.Series(label_key).value_counts()
strat = np.array([k if vc[k] >= 2 else "rare" for k in label_key])

try:
    tr_idx, va_idx = train_test_split(
        np.arange(len(train)),
        test_size=0.18,
        random_state=CFG["seed"],
        stratify=strat
    )
except Exception:
    tr_idx, va_idx = train_test_split(
        np.arange(len(train)),
        test_size=0.18,
        random_state=CFG["seed"],
        stratify=None
    )

print("train split:", len(tr_idx), "valid split:", len(va_idx))
print("valid label mean:", Y[va_idx].mean(axis=0))

train split: 13694 valid split: 3007
valid label mean: [0.42800133 0.13601596 0.11040905 0.07449285 0.03259062]


In [ ]:
def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def hamming_score(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    return hamming_loss(y_true, pred)

def tune_thresholds(y_true, prob, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)
    thresholds = np.zeros(y_true.shape[1], dtype=float)
    for j in range(y_true.shape[1]):
        best_t, best_s = 0.5, 10**9
        for t in grid:
            s = hamming_loss(y_true[:, [j]], (prob[:, [j]] >= t).astype(int))
            if s < best_s:
                best_s = s
                best_t = t
        thresholds[j] = best_t
    return thresholds

def tune_global_and_class_thresholds(y_true, prob):
    best_g, best_s = 0.5, 10**9
    for t in np.linspace(0.05, 0.95, 91):
        s = hamming_loss(y_true, (prob >= t).astype(int))
        if s < best_s:
            best_s, best_g = s, t
    class_thr = tune_thresholds(y_true, prob)
    class_s = hamming_loss(y_true, (prob >= class_thr).astype(int))
    if class_s <= best_s:
        return class_thr, class_s
    return np.ones(y_true.shape[1]) * best_g, best_s

def greedy_ensemble(y_true, valid_preds, test_preds, max_steps=50):
    names = list(valid_preds.keys())
    if len(names) == 0:
        raise ValueError("Нет предсказаний для ансамбля")

    best_name = None
    best_score = 10**9
    best_thr = None
    for name in names:
        thr, score = tune_global_and_class_thresholds(y_true, valid_preds[name])
        print(f"single {name:35s} score={score:.6f} thr={np.round(thr, 3)}")
        if score < best_score:
            best_score = score
            best_name = name
            best_thr = thr

    ens_valid = valid_preds[best_name].copy()
    ens_test = test_preds[best_name].copy()
    chosen = [best_name]
    scores = [best_score]

    print("\nstart ensemble:", best_name, best_score)
    for step in range(max_steps):
        improved = False
        step_best = None
        step_score = best_score
        step_thr = best_thr

        for name in names:
            candidate_valid = (ens_valid * len(chosen) + valid_preds[name]) / (len(chosen) + 1)
            thr, score = tune_global_and_class_thresholds(y_true, candidate_valid)
            if score + 1e-12 < step_score:
                step_score = score
                step_best = name
                step_thr = thr
                improved = True

        if not improved:
            break

        chosen.append(step_best)
        ens_valid = (ens_valid * (len(chosen) - 1) + valid_preds[step_best]) / len(chosen)
        ens_test = (ens_test * (len(chosen) - 1) + test_preds[step_best]) / len(chosen)
        best_score = step_score
        best_thr = step_thr
        scores.append(best_score)
        print(f"step {step+1:02d}: add {step_best:35s} score={best_score:.6f}")

    weights = Counter(chosen)
    weights = {k: v / len(chosen) for k, v in weights.items()}
    print("\nfinal weights:", weights)
    print("final valid score:", best_score)
    print("final thresholds:", np.round(best_thr, 4))
    return ens_valid, ens_test, best_thr, weights, scores

In [ ]:
valid_preds = {}
test_preds = {}
model_scores = {}

def make_sparse_features(train_text, test_text):
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        strip_accents=None,
        max_features=350_000,
        dtype=np.float32,
    )
    char = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 6),
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
        max_features=450_000,
        dtype=np.float32,
    )
    all_text = pd.concat([train_text, test_text], axis=0)
    word.fit(all_text)
    char.fit(all_text)
    Xtr = sparse.hstack([word.transform(train_text), char.transform(train_text)], format="csr")
    Xte = sparse.hstack([word.transform(test_text), char.transform(test_text)], format="csr")
    return Xtr, Xte

if CFG["use_sparse"]:
    print("building sparse features...")
    X_sparse, X_test_sparse = make_sparse_features(train_feat["full_text"], test_feat["full_text"])
    Xtr_sp = X_sparse[tr_idx]
    Xva_sp = X_sparse[va_idx]
    Ytr = np.array(Y[tr_idx], dtype=np.int32).copy()
    Yva = np.array(Y[va_idx], dtype=np.int32).copy()

    sparse_models = []

    sparse_models.append((
        "tfidf_lr_c2",
        OneVsRestClassifier(
            LogisticRegression(
                C=2.0,
                solver="liblinear",
                class_weight=None,
                max_iter=2000,
                random_state=CFG["seed"],
            ),
            n_jobs=1
        )
    ))

    sparse_models.append((
        "tfidf_lr_balanced",
        OneVsRestClassifier(
            LogisticRegression(
                C=1.4,
                solver="liblinear",
                class_weight="balanced",
                max_iter=2000,
                random_state=CFG["seed"] + 1,
            ),
            n_jobs=1
        )
    ))

    sparse_models.append((
        "tfidf_sgd_huber",
        OneVsRestClassifier(
            SGDClassifier(
                loss="modified_huber",
                alpha=1e-5,
                penalty="l2",
                max_iter=2000,
                tol=1e-4,
                random_state=CFG["seed"] + 2,
            ),
            n_jobs=1
        )
    ))

    sparse_models.append((
        "tfidf_nb",
        OneVsRestClassifier(
            ComplementNB(alpha=0.15),
            n_jobs=1
        )
    ))

    sparse_models.append((
        "tfidf_chain_lr",
        ClassifierChain(
            LogisticRegression(
                C=1.2,
                solver="liblinear",
                max_iter=1500,
                random_state=CFG["seed"] + 3,
            ),
            order="random",
            random_state=CFG["seed"] + 4,
        )
    ))

    for name, model in sparse_models:
        print("\ntraining", name)
        try:
            model.fit(Xtr_sp, Ytr)
            if hasattr(model, "predict_proba"):
                pv = model.predict_proba(Xva_sp)
                pt = model.predict_proba(X_test_sparse)
                if isinstance(pv, list):
                    pv = np.vstack([p[:, 1] for p in pv]).T
                    pt = np.vstack([p[:, 1] for p in pt]).T
            else:
                dv = model.decision_function(Xva_sp)
                dt = model.decision_function(X_test_sparse)
                pv, pt = sigmoid_np(dv), sigmoid_np(dt)
            pv = np.asarray(pv, dtype=np.float32)
            pt = np.asarray(pt, dtype=np.float32)
            thr, score = tune_global_and_class_thresholds(Yva, pv)
            print(name, "valid hamming:", score, "thr:", np.round(thr, 3))
            valid_preds[name] = pv
            test_preds[name] = pt
            model_scores[name] = float(score)
        except Exception as e:
            print("FAILED", name, repr(e))

    print("\nbuilding title sparse features...")
    X_title, X_test_title = make_sparse_features(train_feat["title_only"], test_feat["title_only"])
    title_model = OneVsRestClassifier(
        LogisticRegression(C=1.0, solver="liblinear", max_iter=1500, random_state=CFG["seed"]+10),
        n_jobs=1
    )
    title_model.fit(X_title[tr_idx], Ytr)
    pv = title_model.predict_proba(X_title[va_idx]).astype(np.float32)
    pt = title_model.predict_proba(X_test_title).astype(np.float32)
    thr, score = tune_global_and_class_thresholds(Yva, pv)
    print("title_lr valid hamming:", score, "thr:", np.round(thr, 3))
    valid_preds["title_lr"] = pv
    test_preds["title_lr"] = pt
    model_scores["title_lr"] = float(score)

    del Xtr_sp, Xva_sp
    gc.collect()

building sparse features...

training tfidf_lr_c2
tfidf_lr_c2 valid hamming: 0.05034918523445294 thr: [0.46 0.29 0.33 0.32 0.45]

training tfidf_lr_balanced
tfidf_lr_balanced valid hamming: 0.050748254073827735 thr: [0.5  0.6  0.65 0.7  0.89]

training tfidf_sgd_huber
tfidf_sgd_huber valid hamming: 0.05819753907549052 thr: [0.55 0.5  0.34 0.47 0.4 ]

training tfidf_nb
tfidf_nb valid hamming: 0.05906218822746924 thr: [0.22 0.22 0.72 0.05 0.07]

training tfidf_chain_lr
tfidf_chain_lr valid hamming: 0.050149650814765544 thr: [0.47 0.36 0.31 0.32 0.36]

building title sparse features...
title_lr valid hamming: 0.07010309278350516 thr: [0.49 0.33 0.26 0.24 0.18]


In [ ]:
if TORCH_OK and CFG["use_textcnn"]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    TOKEN_RE = re.compile(r"[а-яёa-z0-9]+|[%.,!?;:/()+-]", flags=re.I)

    def tokenize_for_nn(text):
        return TOKEN_RE.findall(str(text).lower())

    def build_vocab(texts, max_size=70000, min_freq=2):
        cnt = Counter()
        for t in texts:
            cnt.update(tokenize_for_nn(t))
        vocab = {"<pad>": 0, "<unk>": 1}
        for w, c in cnt.most_common(max_size - 2):
            if c >= min_freq:
                vocab[w] = len(vocab)
        return vocab

    vocab = build_vocab(train_feat["full_text"].iloc[tr_idx], CFG["textcnn_vocab_size"], min_freq=2)
    print("vocab size:", len(vocab))

    def encode_text(text, vocab, max_len):
        toks = tokenize_for_nn(text)
        ids = [vocab.get(t, 1) for t in toks[:max_len]]
        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))
        return np.array(ids, dtype=np.int64)

    class TextDataset(Dataset):
        def __init__(self, texts, labels=None, vocab=None, max_len=700):
            self.texts = list(texts)
            self.labels = labels
            self.vocab = vocab
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            x = encode_text(self.texts[idx], self.vocab, self.max_len)
            item = {"input_ids": torch.tensor(x, dtype=torch.long)}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
            return item

    class AttentionPool(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.w = nn.Sequential(
                nn.Linear(dim, dim),
                nn.Tanh(),
                nn.Linear(dim, 1)
            )
        def forward(self, x, mask=None):
            scores = self.w(x).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(mask == 0, -1e9)
            a = torch.softmax(scores, dim=1).unsqueeze(-1)
            return (x * a).sum(dim=1)

    class TextCNNBiGRU(nn.Module):
        def __init__(self, vocab_size, n_labels=5, emb_dim=256, hidden=128, dropout=0.35):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            self.emb_dropout = nn.Dropout(0.20)
            self.convs = nn.ModuleList([
                nn.Conv1d(emb_dim, 192, k, padding=k//2) for k in [2, 3, 4, 5]
            ])
            self.gru = nn.GRU(
                emb_dim,
                hidden,
                batch_first=True,
                bidirectional=True,
                num_layers=1
            )
            self.attn = AttentionPool(hidden * 2)
            total_dim = 192 * 4 + hidden * 2
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Sequential(
                nn.Linear(total_dim, 384),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(384, n_labels)
            )
        def forward(self, input_ids):
            mask = (input_ids != 0).long()
            x = self.emb(input_ids)
            x = self.emb_dropout(x)

            xc = x.transpose(1, 2)
            conv_outs = []
            for conv in self.convs:
                y = F.relu(conv(xc))
                y = F.adaptive_max_pool1d(y, 1).squeeze(-1)
                conv_outs.append(y)
            cnn_feat = torch.cat(conv_outs, dim=1)

            gru_out, _ = self.gru(x)
            gru_feat = self.attn(gru_out, mask)

            feat = torch.cat([cnn_feat, gru_feat], dim=1)
            return self.fc(self.drop(feat))

    def train_textcnn(train_indices, valid_indices, full_fit=False):
        seed_everything(CFG["seed"] + (111 if full_fit else 0))
        if full_fit:
            train_indices = np.arange(len(train_feat))
            valid_indices = None

        X_text = train_feat["full_text"].values
        y_arr = Y.astype(np.float32)

        train_ds = TextDataset(X_text[train_indices], y_arr[train_indices], vocab, CFG["textcnn_max_len"])
        train_loader = DataLoader(
            train_ds,
            batch_size=CFG["textcnn_batch_size"],
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
        if valid_indices is not None:
            valid_ds = TextDataset(X_text[valid_indices], y_arr[valid_indices], vocab, CFG["textcnn_max_len"])
            valid_loader = DataLoader(valid_ds, batch_size=CFG["textcnn_batch_size"]*2, shuffle=False, num_workers=0)
        else:
            valid_loader = None

        model = TextCNNBiGRU(
            vocab_size=len(vocab),
            n_labels=CFG["n_labels"],
            emb_dim=CFG["textcnn_emb_dim"],
        ).to(device)

        pos = y_arr[train_indices].mean(axis=0)
        pos_weight = torch.tensor((1 - pos) / np.clip(pos, 1e-4, 1), dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["textcnn_lr"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["textcnn_epochs"] * len(train_loader)))

        best_state = None
        best_score = 10**9

        for epoch in range(CFG["textcnn_epochs"]):
            model.train()
            losses = []
            for batch in train_loader:
                ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                optimizer.zero_grad()
                logits = model(ids)
                loss = criterion(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                losses.append(float(loss.detach().cpu()))

            msg = f"epoch {epoch+1}/{CFG['textcnn_epochs']} loss={np.mean(losses):.5f}"
            if valid_loader is not None:
                pv = predict_textcnn(model, valid_loader)
                thr, score = tune_global_and_class_thresholds(Y[valid_indices], pv)
                msg += f" valid_hamming={score:.6f}"
                if score < best_score:
                    best_score = score
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(msg)

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    @torch.no_grad()
    def predict_textcnn(model, loader):
        model.eval()
        preds = []
        for batch in loader:
            ids = batch["input_ids"].to(device)
            logits = model(ids)
            preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        return np.vstack(preds).astype(np.float32)

    print("\ntraining TextCNN+BiGRU...")
    textcnn_model = train_textcnn(tr_idx, va_idx, full_fit=False)

    valid_ds = TextDataset(train_feat["full_text"].values[va_idx], None, vocab, CFG["textcnn_max_len"])
    test_ds = TextDataset(test_feat["full_text"].values, None, vocab, CFG["textcnn_max_len"])
    valid_loader = DataLoader(valid_ds, batch_size=CFG["textcnn_batch_size"]*2, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CFG["textcnn_batch_size"]*2, shuffle=False, num_workers=0)

    pv = predict_textcnn(textcnn_model, valid_loader)
    pt = predict_textcnn(textcnn_model, test_loader)
    thr, score = tune_global_and_class_thresholds(Y[va_idx], pv)
    print("textcnn_bigru valid hamming:", score, "thr:", np.round(thr, 3))
    valid_preds["textcnn_bigru"] = pv
    test_preds["textcnn_bigru"] = pt
    model_scores["textcnn_bigru"] = float(score)

    del valid_loader, test_loader, valid_ds, test_ds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

device: cuda
vocab size: 70000

training TextCNN+BiGRU...
epoch 1/6 loss=1.08748 valid_hamming=0.117260
epoch 2/6 loss=0.93534 valid_hamming=0.110409
epoch 3/6 loss=0.82590 valid_hamming=0.098969
epoch 4/6 loss=0.73827 valid_hamming=0.098304
epoch 5/6 loss=0.67276 valid_hamming=0.095377
epoch 6/6 loss=0.63995 valid_hamming=0.095976
textcnn_bigru valid hamming: 0.09537745261057533 thr: [0.62 0.89 0.95 0.94 0.93]


In [ ]:
if TORCH_OK and TRANSFORMERS_OK and CFG["use_hf"]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    class HFDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length):
            self.texts = list(texts)
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_length = max_length
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors=None
            )
            item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
            return item

    def load_hf_model(model_cfg):
        kwargs = {
            "num_labels": CFG["n_labels"],
            "problem_type": "multi_label_classification",
            "cache_dir": str(HF_HUB_CACHE),
            "use_safetensors": model_cfg.get("use_safetensors", True),
        }
        if model_cfg.get("revision"):
            kwargs["revision"] = model_cfg["revision"]

        tokenizer_kwargs = {"cache_dir": str(HF_HUB_CACHE)}
        if model_cfg.get("revision"):
            tokenizer_kwargs["revision"] = model_cfg["revision"]

        tokenizer = AutoTokenizer.from_pretrained(model_cfg["name"], **tokenizer_kwargs)
        model = AutoModelForSequenceClassification.from_pretrained(model_cfg["name"], **kwargs)
        return tokenizer, model

    @torch.no_grad()
    def predict_hf(model, loader):
        model.eval()
        preds = []
        for batch in loader:
            labels = batch.pop("labels", None)
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds.append(torch.sigmoid(out.logits).detach().cpu().numpy())
        return np.vstack(preds).astype(np.float32)

    def train_hf_on_split(model_cfg, train_indices, valid_indices):
        print("\nHF model:", model_cfg["name"])
        seed_everything(CFG["seed"])
        tokenizer, model = load_hf_model(model_cfg)
        model.to(device)

        y_float = Y.astype(np.float32)
        tr_ds = HFDataset(
            train_feat["short_text"].values[train_indices],
            y_float[train_indices],
            tokenizer,
            model_cfg["max_length"]
        )
        va_ds = HFDataset(
            train_feat["short_text"].values[valid_indices],
            y_float[valid_indices],
            tokenizer,
            model_cfg["max_length"]
        )
        te_ds = HFDataset(
            test_feat["short_text"].values,
            None,
            tokenizer,
            model_cfg["max_length"]
        )

        tr_loader = DataLoader(tr_ds, batch_size=model_cfg["batch_size"], shuffle=True, num_workers=0)
        va_loader = DataLoader(va_ds, batch_size=model_cfg["valid_batch_size"], shuffle=False, num_workers=0)
        te_loader = DataLoader(te_ds, batch_size=model_cfg["valid_batch_size"], shuffle=False, num_workers=0)

        optimizer = torch.optim.AdamW(model.parameters(), lr=model_cfg["lr"], weight_decay=model_cfg["weight_decay"])
        total_steps = max(1, len(tr_loader) * model_cfg["epochs"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

        best_state = None
        best_score = 10**9

        for epoch in range(model_cfg["epochs"]):
            model.train()
            losses = []
            for batch in tr_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                out = model(**batch)
                loss = out.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                losses.append(float(loss.detach().cpu()))

            pv = predict_hf(model, va_loader)
            thr, score = tune_global_and_class_thresholds(Y[valid_indices], pv)
            print(f"{model_cfg['tag']} epoch {epoch+1}/{model_cfg['epochs']} loss={np.mean(losses):.5f} valid={score:.6f}")
            if score < best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state is not None:
            model.load_state_dict(best_state)
        pv = predict_hf(model, va_loader)
        pt = predict_hf(model, te_loader)

        del tr_loader, va_loader, te_loader, tr_ds, va_ds, te_ds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return pv, pt

    for model_cfg in CFG["hf_models"]:
        try:
            pv, pt = train_hf_on_split(model_cfg, tr_idx, va_idx)
            thr, score = tune_global_and_class_thresholds(Y[va_idx], pv)
            name = "hf_" + model_cfg["tag"]
            print(name, "valid hamming:", score, "thr:", np.round(thr, 3))
            valid_preds[name] = pv
            test_preds[name] = pt
            model_scores[name] = float(score)
        except Exception as e:
            print("HF model failed, skip:", model_cfg["name"], repr(e))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

device: cuda

HF model: cointegrated/rubert-tiny2


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 13751.82it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

rubert_tiny2 epoch 1/4 loss=0.29902 valid=0.058530
rubert_tiny2 epoch 2/4 loss=0.17859 valid=0.049950
rubert_tiny2 epoch 3/4 loss=0.15151 valid=0.049684
rubert_tiny2 epoch 4/4 loss=0.14248 valid=0.049884
hf_rubert_tiny2 valid hamming: 0.04968407050216162 thr: [0.5  0.48 0.51 0.37 0.39]

HF model: FacebookAI/xlm-roberta-base


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5395.15it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


xlm_roberta_base epoch 1/3 loss=0.19428 valid=0.048487
xlm_roberta_base epoch 2/3 loss=0.13706 valid=0.044430
xlm_roberta_base epoch 3/3 loss=0.10921 valid=0.045161
hf_xlm_roberta_base valid hamming: 0.044429664117060196 thr: [0.47 0.4  0.52 0.27 0.29]


In [ ]:
dupe_valid = np.full((len(va_idx), CFG["n_labels"]), np.nan, dtype=np.float32)
dupe_test = np.full((len(test), CFG["n_labels"]), np.nan, dtype=np.float32)

label_by_text = {}
for i in tr_idx:
    key = train_feat["clean_title"].iloc[i] + " || " + train_feat["clean_text"].iloc[i]
    if key not in label_by_text:
        label_by_text[key] = []
    label_by_text[key].append(Y[i])

label_by_text = {k: np.mean(v, axis=0) for k, v in label_by_text.items()}

for pos, i in enumerate(va_idx):
    key = train_feat["clean_title"].iloc[i] + " || " + train_feat["clean_text"].iloc[i]
    if key in label_by_text:
        dupe_valid[pos] = label_by_text[key]

for i in range(len(test)):
    key = test_feat["clean_title"].iloc[i] + " || " + test_feat["clean_text"].iloc[i]
    if key in label_by_text:
        dupe_test[i] = label_by_text[key]

valid_dupe_mask = ~np.isnan(dupe_valid).any(axis=1)
test_dupe_mask = ~np.isnan(dupe_test).any(axis=1)
print("valid exact duplicates:", valid_dupe_mask.sum())
print("test exact duplicates:", test_dupe_mask.sum())

if valid_dupe_mask.sum() > 0:
    base_valid = np.mean(list(valid_preds.values()), axis=0)
    base_test = np.mean(list(test_preds.values()), axis=0)
    base_valid[valid_dupe_mask] = dupe_valid[valid_dupe_mask]
    base_test[test_dupe_mask] = dupe_test[test_dupe_mask]
    valid_preds["exact_duplicate_booster"] = base_valid.astype(np.float32)
    test_preds["exact_duplicate_booster"] = base_test.astype(np.float32)
    thr, score = tune_global_and_class_thresholds(Y[va_idx], base_valid)
    model_scores["exact_duplicate_booster"] = float(score)
    print("exact_duplicate_booster valid hamming:", score)

valid exact duplicates: 0
test exact duplicates: 0


In [ ]:
print("models collected:", list(valid_preds.keys()))
if len(valid_preds) == 0:
    raise RuntimeError("Ни одна модель не обучилась. Проверь зависимости и данные.")

ens_valid, ens_test, thresholds, weights, ens_scores = greedy_ensemble(
    Y[va_idx],
    valid_preds,
    test_preds,
    max_steps=40
)

final_valid_score = hamming_loss(Y[va_idx], (ens_valid >= thresholds).astype(int))
print("FINAL VALID HAMMING:", final_valid_score)
print("thresholds:", thresholds)

models collected: ['tfidf_lr_c2', 'tfidf_lr_balanced', 'tfidf_sgd_huber', 'tfidf_nb', 'tfidf_chain_lr', 'title_lr', 'textcnn_bigru', 'hf_rubert_tiny2', 'hf_xlm_roberta_base']
single tfidf_lr_c2                         score=0.050349 thr=[0.46 0.29 0.33 0.32 0.45]
single tfidf_lr_balanced                   score=0.050748 thr=[0.5  0.6  0.65 0.7  0.89]
single tfidf_sgd_huber                     score=0.058198 thr=[0.55 0.5  0.34 0.47 0.4 ]
single tfidf_nb                            score=0.059062 thr=[0.22 0.22 0.72 0.05 0.07]
single tfidf_chain_lr                      score=0.050150 thr=[0.47 0.36 0.31 0.32 0.36]
single title_lr                            score=0.070103 thr=[0.49 0.33 0.26 0.24 0.18]
single textcnn_bigru                       score=0.095377 thr=[0.62 0.89 0.95 0.94 0.93]
single hf_rubert_tiny2                     score=0.049684 thr=[0.5  0.48 0.51 0.37 0.39]
single hf_xlm_roberta_base                 score=0.044430 thr=[0.47 0.4  0.52 0.27 0.29]

start ensemble: hf_xlm_

In [ ]:
test_pred = (ens_test >= thresholds).astype(int)

if test_dupe_mask.sum() > 0:
    test_pred[test_dupe_mask] = (dupe_test[test_dupe_mask] >= 0.5).astype(int)

submission = sample.copy()
submission["target"] = ["[" + ",".join(map(str, row.tolist())) + "]" for row in test_pred.astype(int)]
submission.to_csv(CFG["out_csv"], index=False)
submission.to_csv(CFG["extra_out_csv"], index=False)

info = {
    "student_id": STUDENT_ID,
    "valid_hamming": float(final_valid_score),
    "thresholds": thresholds.tolist(),
    "weights": weights,
    "model_scores": model_scores,
    "n_train": int(len(train)),
    "n_test": int(len(test)),
}
with open(WORK_DIR / "run_info_v6.json", "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("saved:", CFG["out_csv"])
print("saved:", CFG["extra_out_csv"])
print("run info:", WORK_DIR / "run_info_v6.json")
display(submission.head())
print(submission["target"].value_counts().head(20))

saved: D:\ДЛ РАНХ\кагл 2\sample_submission.csv
saved: D:\ДЛ РАНХ\кагл 2\sample_submission_322_v6.csv
run info: D:\dl_kaggle_322_v6\work\run_info_v6.json


,id,target
0,16701,"[0,0,0,0,0]"
1,16702,"[0,0,0,0,0]"
2,16703,"[0,0,0,0,0]"
3,16704,"[0,0,0,0,0]"
4,16705,"[0,0,0,0,0]"


target
[1,0,0,0,0]    2032
[0,0,0,0,0]    1600
[0,0,1,0,0]     501
[0,1,0,0,0]     259
[0,0,0,1,0]     220
[1,1,0,0,0]     110
[1,0,0,1,0]      89
[0,0,0,0,1]      81
[0,1,0,1,0]      40
[1,1,0,1,0]      17
[1,0,1,0,0]      14
[1,0,0,0,1]       2
[0,0,1,0,1]       2
[0,0,1,1,0]       2
Name: count, dtype: int64


In [ ]:
check = pd.read_csv(CFG["out_csv"])
assert list(check.columns) == list(sample.columns), (check.columns, sample.columns)
assert len(check) == len(sample)
assert check["id"].equals(sample["id"])
assert check["target"].map(lambda x: bool(re.fullmatch(r"\[[01],[01],[01],[01],[01]\]", str(x)))).all()
print("Формат корректный. Можно отправлять sample_submission.csv")

Формат корректный. Можно отправлять sample_submission.csv
